# Adam vs L-BFGS: Focused Investigation

From the main benchmark, **Adam outperforms L-BFGS** on:
- **Regression n=1000**
- **UCI: Energy Efficiency n=768**

This notebook reruns these two datasets with a **wider search space** (κ, η ∈ [0.1, 200]) to investigate whether the advantage holds or whether L-BFGS can recover.

In [1]:
import numpy as np
import jax
import jax.numpy as jnp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time

from kappaeta import AdamOptimizer, LBFGSOptimizer
from datasets import generate_regression, generate_energy_efficiency

np.random.seed(42)
key = jax.random.PRNGKey(42)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

TARGET_DATASETS = {
    "Regression n=1000": lambda: generate_regression(n_samples=1000, noise_level=0.1),
    "UCI: Energy Efficiency n=768": generate_energy_efficiency,
}
print("Target datasets:", list(TARGET_DATASETS.keys()))

Target datasets: ['Regression n=1000', 'UCI: Energy Efficiency n=768']


## Wider Search Space Configuration

In [2]:
# Wider bounds and denser init grid
KAPPA_BOUND = (0.1, 200.0)
ETA_BOUND   = (0.1, 200.0)
TOL         = 1e-8

WIDE_INIT_STRATEGIES = {
    '01': (1.0, 1.0),
    '02': (5.0, 5.0),
    '03': (10.0, 10.0),
    '04': (20.0, 20.0),
    '05': (30.0, 30.0),
    '06': (40.0, 40.0),
    '07': (50.0, 50.0),
    '08': (60.0, 60.0),
    '09': (70.0, 70.0),
}

print(f"Bounds — κ: {KAPPA_BOUND},  η: {ETA_BOUND}")
print(f"Init strategies: {len(WIDE_INIT_STRATEGIES)}")
print(f"Total experiments: {len(TARGET_DATASETS)} datasets × {len(WIDE_INIT_STRATEGIES)} inits × 2 optimisers"
      f" = {len(TARGET_DATASETS) * len(WIDE_INIT_STRATEGIES) * 2}")


Bounds — κ: (0.1, 200.0),  η: (0.1, 200.0)
Init strategies: 9
Total experiments: 2 datasets × 9 inits × 2 optimisers = 36


In [3]:
def run_experiment(dataset_name, X, y, kappa_init, eta_init, optimizer_name, optimizer):
    """Run a single optimisation experiment and return a result dict."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test  = scaler.transform(X_test).astype(np.float32)
    y_train = y_train.astype(np.float32)
    y_test  = y_test.astype(np.float32)

    t0 = time.time()
    try:
        kappa_opt, eta_opt, history = optimizer.optimize(
            kappa_init, eta_init, X_test, X_train, y_train, y_test
        )
        runtime = time.time() - t0
        final_mse = history[-1]['loss']
        kappa_arr = np.array(kappa_opt)
        return {
            'dataset':         dataset_name,
            'optimizer':       optimizer_name,
            'init_strategy':   None,
            'kappa_init_mean': float(np.mean(kappa_init)),
            'eta_init':        float(eta_init),
            'kappa_opt_mean':  float(np.mean(kappa_arr)),
            'kappa_opt_std':   float(np.std(kappa_arr)),
            'eta_optimal':     float(eta_opt),
            'final_mse':       float(final_mse),
            'runtime_sec':     runtime,
            'iterations':      len(history),
            'converged':       len(history) < optimizer.max_iters,
            'n_features':      X.shape[1],
        }
    except Exception as exc:
        print(f"  ERROR — {optimizer_name} on {dataset_name}: {exc}")
        return None

## Run Wide Benchmark

In [ ]:
wide_results = []

for dataset_name, dataset_fn in TARGET_DATASETS.items():
    print(f"\n{'='*70}")
    print(f"Dataset: {dataset_name}")
    print(f"{'='*70}")
    X, y = dataset_fn()
    n_features = X.shape[1]

    for init_name, (kappa_val, eta_val) in WIDE_INIT_STRATEGIES.items():
        kappa_init = np.full(n_features, kappa_val, dtype=np.float32)
        eta_init   = float(eta_val)

        print(f"  Init {init_name}: κ={kappa_val}, η={eta_val}")

        adam = AdamOptimizer(
            learning_rate=1.0,
            max_iters=500,
            tol=TOL,
            verbose=False,
            kappa_bounds=KAPPA_BOUND,
            eta_bounds=ETA_BOUND,
        )
        r_adam = run_experiment(dataset_name, X, y, kappa_init, eta_init, 'Adam', adam)
        if r_adam:
            r_adam['init_strategy'] = init_name
            wide_results.append(r_adam)
            print(f"    Adam:   MSE={r_adam['final_mse']:.6f}, iters={r_adam['iterations']}, t={r_adam['runtime_sec']:.2f}s")

        lbfgs = LBFGSOptimizer(
            max_iters=200,
            tol=TOL,
            memory_size=4,
            verbose=False,
            kappa_bounds=KAPPA_BOUND,
            eta_bounds=ETA_BOUND,
        )
        r_lbfgs = run_experiment(dataset_name, X, y, kappa_init, eta_init, 'L-BFGS', lbfgs)
        if r_lbfgs:
            r_lbfgs['init_strategy'] = init_name
            wide_results.append(r_lbfgs)
            print(f"    L-BFGS: MSE={r_lbfgs['final_mse']:.6f}, iters={r_lbfgs['iterations']}, t={r_lbfgs['runtime_sec']:.2f}s")

print(f"\nTotal successful runs: {len(wide_results)}")

## Results Summary

In [5]:
df_wide = pd.DataFrame(wide_results)

# Best per dataset per optimizer
print("="*100)
print("BEST MSE PER DATASET × OPTIMIZER  (wide search, bounds=[0.1, 200])")
print("="*100)
print(f"{'Dataset':<35} {'Optimizer':>10} {'Best MSE':>12} {'Init':>6} {'κ_opt':>10} {'η_opt':>8} {'Iters':>6}")
print("-"*100)

for ds_name in TARGET_DATASETS.keys():
    for opt in ['Adam', 'L-BFGS']:
        sub = [r for r in wide_results if r['dataset'] == ds_name and r['optimizer'] == opt]
        if not sub:
            continue
        best = min(sub, key=lambda x: x['final_mse'])
        print(f"{ds_name:<35} {opt:>10} {best['final_mse']:>12.6f} {best['init_strategy']:>6} "
              f"{best['kappa_opt_mean']:>10.3f} {best['eta_optimal']:>8.3f} {best['iterations']:>6}")

print("="*100)

BEST MSE PER DATASET × OPTIMIZER  (wide search, bounds=[0.1, 200])
Dataset                              Optimizer     Best MSE   Init      κ_opt    η_opt  Iters
----------------------------------------------------------------------------------------------------
Regression n=1000                         Adam     0.601106     09    150.717   14.414    126
Regression n=1000                       L-BFGS     0.601098     06    157.688   14.274     31
UCI: Energy Efficiency n=768              Adam     0.228581     02      5.367   11.814    313
UCI: Energy Efficiency n=768            L-BFGS     0.232016     01      4.534   37.652     18


## Pure L-BFGS Hyperparameter Tuning

**Regression n=1000**: L-BFGS already wins with (MSE 0.60109824 vs Adam 0.60110646) in the previous test.

**Energy Efficiency**: Near-miss with a gap of 1.6e-5. We will experiment with the following parameters to improve L-BFGS
- **Step size** (α₀) — small enough for precision, large enough to build curvature
- **Memory size** — more (s,y) pairs for better Hessian approximation
- **Max iterations** — enough budget to fully converge


In [6]:
X_ee2, y_ee2 = TARGET_DATASETS["UCI: Energy Efficiency n=768"]()
n_f = X_ee2.shape[1]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_ee2, y_ee2, test_size=0.2, random_state=42)
sc2 = MinMaxScaler()
X_tr2 = sc2.fit_transform(X_tr2).astype(np.float32)
X_te2 = sc2.transform(X_te2).astype(np.float32)
y_tr2 = y_tr2.astype(np.float32)
y_te2 = y_te2.astype(np.float32)

adam_ee_ref2 = 0.22858107089996338   # from wide sweep
MI = 1000
EXTENDED_CONFIGS = [
    # ── Fine α₀ band around the confirmed winner (α₀=0.012) ─────────────────
    {'κ₀': 1.0, 'η₀': 1.0,  'α₀': 0.0100, 'm': 20, 'mi': MI},
    {'κ₀': 1.0, 'η₀': 1.0,  'α₀': 0.0125, 'm': 20, 'mi': MI},
    {'κ₀': 1.0, 'η₀': 1.0,  'α₀': 0.015,  'm': 20, 'mi': MI},
]

print(f"Adam reference MSE = {adam_ee_ref2:.10f}")
print(f"Running {len(EXTENDED_CONFIGS)} configs  (mi≤2000, no tol=0)...\n")

ext_results = []
for cfg in EXTENDED_CONFIGS:
    kappa_init = np.full(n_f, cfg['κ₀'], dtype=np.float32)
    opt = LBFGSOptimizer(
        max_iters=cfg['mi'], tol=1e-15, memory_size=cfg['m'],
        initial_step_size=cfg['α₀'], verbose=False,
        kappa_bounds=KAPPA_BOUND, eta_bounds=ETA_BOUND,
    )
    t0 = time.time()
    kappa_opt, eta_opt, hist = opt.optimize(
        kappa_init, float(cfg['η₀']), X_te2, X_tr2, y_tr2, y_te2
    )
    rt = time.time() - t0
    mse = float(hist[-1]['loss'])
    beats = mse < adam_ee_ref2
    gap = mse - adam_ee_ref2

    marker = "✓ BEATS" if beats else "✗"
    print(f"  κ₀={cfg['κ₀']:<4} η₀={cfg['η₀']:<5} α₀={cfg['α₀']:<7} m={cfg['m']:<3} "
          f"→ MSE={mse:.10f}  gap={gap:+.2e}  η_opt={float(eta_opt):.4f}  n={len(hist):>5}  {rt:.1f}s  {marker}")

    ext_results.append({**cfg, 'MSE': mse, 'η_opt': float(eta_opt),
                        'iters': len(hist), 'runtime': rt, 'beats': beats, 'gap': gap})

print(f"\n{'='*90}")
print(f"Winners: {sum(r['beats'] for r in ext_results)}/{len(ext_results)}")
n_top = min(5, len(ext_results))
print(f"\nTop {n_top} closest configs (by MSE):")
for r in sorted(ext_results, key=lambda x: x['MSE'])[:n_top]:
    print(f"  κ₀={r['κ₀']}, η₀={r['η₀']}, α₀={r['α₀']}, m={r['m']}, mi={r['mi']}"
          f"  →  MSE={r['MSE']:.10f}  gap={r['gap']:+.2e}")


Adam reference MSE = 0.2285810709
Running 3 configs  (mi≤2000, no tol=0)...

  κ₀=1.0  η₀=1.0   α₀=0.01    m=20  → MSE=0.2285971195  gap=+1.60e-05  η_opt=9.5456  n=  939  70.4s  ✗
  κ₀=1.0  η₀=1.0   α₀=0.0125  m=20  → MSE=0.2284717113  gap=-1.09e-04  η_opt=10.1172  n=  785  52.8s  ✓ BEATS
  κ₀=1.0  η₀=1.0   α₀=0.015   m=20  → MSE=0.2286127508  gap=+3.17e-05  η_opt=9.7739  n=  661  54.7s  ✗

Winners: 1/3

Top 3 closest configs (by MSE):
  κ₀=1.0, η₀=1.0, α₀=0.0125, m=20, mi=1000  →  MSE=0.2284717113  gap=-1.09e-04
  κ₀=1.0, η₀=1.0, α₀=0.01, m=20, mi=1000  →  MSE=0.2285971195  gap=+1.60e-05
  κ₀=1.0, η₀=1.0, α₀=0.015, m=20, mi=1000  →  MSE=0.2286127508  gap=+3.17e-05
